# linalg-solve-batched — worked example 3: Solve K systems with M right-hand sides each — shape (K, n, M)

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `linalg-solve-batched`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

When `b` has shape `(K, n, M)`, `torch.linalg.solve` treats the last dimension as `M` independent right-hand sides per system and returns `x: (K, n, M)`. Each of the K LU factorizations is reused across all M columns, making this significantly faster than looping over columns.

## Worked solution

Step 1: Build `A: (K, n, n)` and `B: (K, n, M)`. `B` is just K small identity-scaled systems.

Step 2: Call `X = t.linalg.solve(A, B)`. The output shape is `(K, n, M)`.

Step 3: Verify: for each `k` and each column `m`, `A[k] @ X[k, :, m]` should equal `B[k, :, m]`. This can be checked in one line with `torch.allclose(A @ X, B)`.

Step 4: Compare timing (conceptually) against a loop over M that would call `solve` M times.

In [ ]:
import torch as t

t.manual_seed(3)

K, n, M = 4, 3, 5

# Random well-conditioned systems
A = t.randn(K, n, n)
A = A + 3 * t.eye(n)   # ensure invertibility

# M different right-hand sides per system
B = t.randn(K, n, M)

# Solve all K*M systems in one call
X = t.linalg.solve(A, B)
print(f'X shape: {X.shape}')   # (4, 3, 5)

# Verify: A @ X should equal B for every (k, m)
residual = (A @ X - B).abs().max().item()
print(f'max residual: {residual:.2e}')
assert residual < 1e-4

# Compare: loop over columns (less efficient)
X_loop = t.stack([t.linalg.solve(A, B[:, :, m]) for m in range(M)], dim=-1)
print(f'X_loop shape: {X_loop.shape}')  # (4, 3, 5)
assert t.allclose(X, X_loop, atol=1e-5), 'loop and batched results differ'
print('Batched and loop results match.')